# Tweet analysis with PySpark RDDs

Cleaned portfolio version of an academic team project. The notebook uses approximately 58,000 tweet records to demonstrate tokenisation, key-value aggregation, joins and set-style transformations. Sentiment is lexicon-based, not a machine-learning classifier.


# **Setting**

In [ ]:
# Install PySpark once if it is not already available.
# %pip install -q pyspark==3.5.1

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName(
    "Tweet communication analysis"
).getOrCreate()
sc = spark.sparkContext

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")


In [ ]:
# Reuse the SparkContext created above.
print(sc.version)


In [ ]:
rdd_tweet = sc.textFile(str(DATA_DIR / "trump_tweets.txt"))

rdd_tweet.take(5)


In [ ]:
# split function

def split_with_multiple_separator1(string_value):
    return re.split('!|"|“|”|:|…|,| |\[|\]|;', string_value.strip())


In [ ]:
# split the words

rdd_tweet_words = rdd_tweet.flatMap(split_with_multiple_separator1).flatMap(lambda word: word.split('.'))\
                                                                   .filter(lambda word: word!='')\
                                                                   .map(lambda word: word.lower())

rdd_tweet_words.take(10)

# **Step 1 : Top 50 references (@) cited**

In [ ]:
rdd_tweet_words_with_reference = rdd_tweet_words.filter(lambda word: '@' in word).filter(lambda word: word!='@')

rdd_tweet_words_with_reference.take(10)

In [ ]:
rdd_tweet_words_with_reference_ordered = rdd_tweet_words_with_reference.map(lambda word: (word, 1))\
                                                                       .reduceByKey(lambda a,b: a+b)\
                                                                       .sortBy(lambda tuple: tuple[1], ascending = False)

list_of_top_reference = rdd_tweet_words_with_reference_ordered.take(50) 

list_of_top_reference

In [ ]:
# make a plot
def plot_reference(list_of_top_reference):
    
    pd_df = pd.DataFrame({
        'Word':  [ t[0] for t in list_of_top_reference ],
        'Count': [ t[1] for t in list_of_top_reference ]
    }).sort_values('Count', ascending=False)
    
    plt.figure( figsize = (15, 8) )
    
    chart=sns.barplot(data=pd_df, x='Word', y='Count', order=pd_df['Word'], color='blue')
    
    chart.set_xticklabels(
        chart.get_xticklabels(), 
        rotation=80, 
        horizontalalignment='right',
        fontweight='light',
        fontsize='x-large',
        )
    
    plt.show()

plot_reference(list_of_top_reference)

We can find more than 10,000 tweets citing @realdonaldtrump, far outstripping any other account, and we can see that trump's popularity has been very high this decade.

# **Step 2 : Top 50 hashtags (#) cited**

In [ ]:
rdd_tweet_words_with_hashtags = rdd_tweet_words.filter(lambda word: '#' in word).filter(lambda word: word!='#')

rdd_tweet_words_with_hashtags.take(10)

In [ ]:
rdd_tweet_words_with_hashtags_ordered = rdd_tweet_words_with_hashtags.map(lambda word: (word, 1))\
                                                                     .reduceByKey(lambda a,b: a+b)\
                                                                     .sortBy(lambda tuple: tuple[1], ascending = False)

list_of_top_hashtags = rdd_tweet_words_with_hashtags_ordered.take(50) 

list_of_top_hashtags

In [ ]:
def plot_hashtags(list_of_top_hashtags):
    
    pd_df = pd.DataFrame({
        'Word':  [ t[0] for t in list_of_top_hashtags ],
        'Count': [ t[1] for t in list_of_top_hashtags ]
    }).sort_values('Count', ascending=False)
    
    plt.figure( figsize = (15, 8) )
    
    chart=sns.barplot(data=pd_df, x='Word', y='Count', order=pd_df['Word'], color='blue')
    
    chart.set_xticklabels(
        chart.get_xticklabels(), 
        rotation=80, 
        horizontalalignment='right',
        fontweight='light',
        fontsize='x-large',
        )
    
    plt.show()

plot_hashtags(list_of_top_hashtags)

We can see that people are very concerned about the election of trump as president in 2016 and the future development of the United States.

# **Step 3 : Top 25 positive and top 25 negative words used**

Step 3.1 : Top 25 positive words

In [ ]:
# initialize an rdd with key/value for counting each word in the file 

rdd_tweet_words_kv = rdd_tweet_words.map(lambda word: (word, 1))

rdd_tweet_words_kv.take(10)

In [ ]:
# read the positive words

rdd_positive_words = sc.textFile(str(DATA_DIR / "positive-words.txt"))

rdd_positive_words.take(5)

In [ ]:
# initialize an rdd with key/value for counting each positive word in the file 


rdd_positive_words_kv = rdd_positive_words.map(lambda word: (word, 1))

rdd_positive_words_kv.take(5)

In [ ]:
# rdd_positive_words_tweet_kv RDD should contain positive words in the tweet review file

rdd_positive_words_tweet_kv = rdd_tweet_words_kv.join(rdd_positive_words_kv)

rdd_positive_words_tweet_kv.take(15)

In [ ]:
# count the top 25 positive words

rdd_positive_words_tweet_kv_ordered = rdd_positive_words_tweet_kv.mapValues(lambda tuple: 1)\
                                                                 .reduceByKey(lambda a,b: a+b)\
                                                                 .sortBy(lambda tuple: tuple[1], ascending = False)

list_top_positives_opinion = rdd_positive_words_tweet_kv_ordered.take(25)

list_top_positives_opinion

Step 3.2 : Top 25 negative words

In [ ]:
rdd_negative_words = sc.textFile(str(DATA_DIR / "negative-words.txt"))

rdd_negative_words.take(5)

In [ ]:
# initialize an rdd with key/value for counting each negative word in the file 

rdd_negative_words_kv = rdd_negative_words.map(lambda word: (word, 1))

rdd_negative_words_kv.take(5)

In [ ]:
# rdd_negative_words_tweet_kv RDD should contain negative words in the tweet review file

rdd_negative_words_tweet_kv = rdd_tweet_words_kv.join(rdd_negative_words_kv)

rdd_negative_words_tweet_kv.take(5)

In [ ]:
rdd_negative_words_tweet_kv_ordered = rdd_negative_words_tweet_kv.mapValues(lambda tuple: 1)\
                                                                 .reduceByKey(lambda a,b: a+b)\
                                                                 .sortBy(lambda tuple: tuple[1], ascending = False)

list_top_negatives_opinion = rdd_negative_words_tweet_kv_ordered.take(25)

list_top_negatives_opinion

Step 3.3 : Plot

In [ ]:
def plot_opinion(list_positives, list_negatives):
    
    list_1 = [(t[0], t[1], 'positive') for t in list_positives]
    list_2 = [(t[0], t[1], 'negative') for t in list_negatives]
    
     
    list_1.extend(list_2)
    
    
    pd_df = pd.DataFrame({
     'Opinion': [ t[0] for t in list_1 ],
     'Count': [ t[1] for t in list_1 ],
     'polarity': [ t[2] for t in list_1 ]
    }).sort_values('Count', ascending=False)
    
    pd_df['color'] = pd_df.polarity.apply(lambda polarity: 'green' if polarity=='positive' else 'red')
    
    plt.figure( figsize = (15, 8) )
    chart=sns.barplot(data=pd_df, x='Opinion', y='Count', hue='polarity', palette=["green", "red"], order=pd_df['Opinion'])
    chart.set_xticklabels(
    chart.get_xticklabels(), 
    rotation=80, 
    horizontalalignment='right',
    fontweight='light',
    fontsize='x-large',
    )
    
    for tick, color in zip(chart.get_xticklabels(), pd_df['color']): 
        tick.set_color(color)
    
    plt.show()

In [ ]:
# call the plot_opinion function to plot our tweets top positive and negative opinions

plot_opinion(list_top_positives_opinion,list_top_negatives_opinion)

On the whole, there are more positive words than negative words. "fake", which ranks first among negative words, reminds people of "fake news", which indicates the high popularity of Trump-related news.

# **Step 4 : Top 50 contextual words**

In [ ]:
rdd_stop_words = sc.textFile(str(DATA_DIR / "stop-words.txt"))

rdd_useless_words = sc.textFile(str(DATA_DIR / "useless-words.txt"))

In [ ]:
def split_with_multiple_separator2(string_value):
    return re.split('!|"|“|”|…|,| |\[|\]|;', string_value.strip())

In [ ]:
new_rdd_tweet_words = rdd_tweet.flatMap(split_with_multiple_separator2).flatMap(lambda word: word.split('.'))\
                                                                       .filter(lambda word: word!='')\
                                                                       .map(lambda word: word.lower())

In [ ]:
# compute the neutral/contextual words RDD by using new_rdd_tweet_words, rdd_positive_words, rdd_negative_words , rdd_stop_words and rdd_useless words
contextual_words = new_rdd_tweet_words.subtract(rdd_positive_words).subtract(rdd_negative_words).subtract(rdd_stop_words)\
                                      .subtract(rdd_tweet_words_with_reference).subtract(rdd_tweet_words_with_hashtags).subtract(rdd_useless_words)

contextual_words.take(5)

In [ ]:
contextual_words_count_ordered = contextual_words.map(lambda word: (word, 1))\
                                                 .reduceByKey(lambda x,y: x+y)\
                                                 .sortBy(lambda tuple: tuple[1], ascending = False)

list_top_contextual_words = contextual_words_count_ordered.take(50)  

list_top_contextual_words

In [ ]:
def plot_contextual_words(list_contextual_words):
    
    pd_df = pd.DataFrame({
        'Word':  [ t[0] for t in list_contextual_words ],
        'Count': [ t[1] for t in list_contextual_words ]
    }).sort_values('Count', ascending=False)
    
    plt.figure( figsize = (15, 8) )
    
    chart=sns.barplot(data=pd_df, x='Word', y='Count', order=pd_df['Word'], color='blue')
    
    chart.set_xticklabels(
        chart.get_xticklabels(), 
        rotation=80, 
        horizontalalignment='right',
        fontweight='light',
        fontsize='x-large',
        )
    
    plt.show()

In [ ]:
plot_contextual_words(list_top_contextual_words)

The top words are "president","people","country","america", which also means people are interested in the topic of the president of the United States.

# **Step 5 : Overall sentiment scores (from positive and negative words)**

In [ ]:
# Compute the global sentiment score
sentiment_score = rdd_positive_words_tweet_kv_ordered.values().sum() - rdd_negative_words_tweet_kv_ordered.values().sum()

print('The sentiment score is : ',sentiment_score)